# Análisis de contractilidad de geles 3D — flujo paso a paso (v4)

**Autores**

&nbsp;&nbsp;&nbsp;&nbsp;Franco Rita

**Descripción**

&nbsp;&nbsp;&nbsp;&nbsp;Este cuaderno recorre, paso a paso y mostrando la salida de cada etapa, el flujo
automatizado que convierte un vídeo de microscopía de un gel 3D en una lista de contracciones
cuantificadas: lectura del vídeo, mapa de máxima intensidad, detección automática de la región
útil de medición (*gauge region*), localización subpíxel de los bordes columna a columna, ajuste
robusto RANSAC que descarta columnas corrompidas, construcción de las series temporales,
**elección del observable**, detección de eventos con verificación estadística del umbral,
medida del adelgazamiento por promedio de eventos alineados, y separación de contracciones
estimuladas y espontáneas por enganche de fase.

---

### Qué cambió en la v4 (2026-09-30)

Cuatro cambios, los cuatro salieron de procesar una batería de videos nuevos y encontrar que el
pipeline v3 fallaba en ellos aunque funcionara en los dos de referencia.

1. **El eje temporal ya no es `fotograma / fps`.** El `fps` que declara un `.mp4` es el
   **promedio**, `(n−1)/duración`, y baja cuando la grabación pierde fotogramas: entonces el eje
   se come los huecos y los eventos aparecen más juntos de lo que fueron. Video_466 perdió el
   4.7 % de sus fotogramas y su período de estimulación medía 9.508 s (un +5.15 % de error contra
   los 0.1 Hz configurados); con los timestamps del contenedor mide 10.006 s. Ahora se usa
   `base_tiempo="pts"`. Detalle en `claude/base-de-tiempo-y-frames-perdidos.md`.
2. **El ancho mínimo de la ROI ya no depende del largo del gel.** Antes era el 35 % de las
   columnas con gel, un valor fijado mirando Video_063. Con geles que ocupan casi todo el cuadro
   eso exige una ROI de ~650 px cuando la *gauge region* real mide 225–470 px, así que la cascada
   se relajaba sola hasta `solo_nitidez` o `franja_completa`. Ahora el mínimo sale de cuántas
   columnas se muestrean (`n_columns × espaciado mínimo`), y hay un nivel de **rescate por
   barrido** para cuando la cascada no encuentra nada plano.
3. **El umbral `k` se elige solo, dentro de la meseta del escaneo de estabilidad.** Antes había
   que mirarlo a ojo en la sección 10. El fallo del `k` por defecto no era ruidoso: en Video_583
   hacía desaparecer el tren estimulado entero sin avisar nada.
4. **Los estimulados se separan de los espontáneos por enganche de fase** (`src/rhythm_split.py`),
   no por amplitud ni por ventana temporal. Sección 12.

### Agregado después (2026-09-29): cinética

**TTP, RT50 y amplitud relativa** (`src/cinetica.py`), en la nueva **sección 11c**. A 30 fps la
subida de las muestras rápidas dura 1–2 fotogramas: ahí se reporta una cota (TTP < 100 ms), no un
valor. Detalle en `claude/metricas-cinetica-TTP-RT50.md`.

### Revisión de consistencia (2026-09-30)

El cuaderno mostraba dos detectores distintos: el del reporte (`scripts/contraction_report.py`, el que
produce `contracciones.xlsx`) y el motor original (`src/event_detection.py`, borrado el 2026-10-01). Daban conteos
distintos en algunos vídeos (29 contra 28 en Video_prueba, 6 contra 5 en Video_466). Además
algunas secciones usaban el fps que declara el archivo (29.x) en vez del de los timestamps (30.000).
Ahora:

- las secciones **8 a 12 usan las mismas funciones que el reporte** (`cr.analizar`), así que lo que
  se ve acá es lo mismo que sale en `contracciones.xlsx`;
- todo el cuaderno usa **el fps de los timestamps**;
- la **sección 12** muestra el enganche de fase de verdad (período, estimulados, espontáneas, p-valor);
- la **sección 13** genera `contracciones.xlsx` llamando al propio script del reporte;
- el canal de detección queda fijo en `center_px` (con `"auto"` el conteo de Video_063 cambiaba de 6 a 8);
- se quitó la "segunda opinión" del motor original, que contradecía a la meseta (en Video_466 decía
  "dudoso" mientras la meseta tenía 0 falsos).

### Lo que sigue valiendo de la v3

- La detección de bordes **no cambió**: mismo subpíxel parabólico con ajuste RANSAC de grado 2 y
  umbral de residuo adaptativo.
- Se guardan los cuatro canales (`y_top_px`, `y_bottom_px`, `thickness_px`, `center_px`), no sólo
  el grosor.
- **La detección se hace sobre `center_px`**, porque en este montaje la contracción es mayormente
  un desplazamiento vertical de toda la franja y el grosor, al ser la resta de los dos bordes, es
  ciego a la traslación.

> **Corrección importante sobre el 13–19 %.** La v3 decía que "sólo un 13–19 % del movimiento es
> cambio de grosor" como si fuera una constante del montaje. Medido sobre seis videos, **depende
> del video**: 18 % en Video_prueba y 13 % en Video_063, pero 1–2 % y **no significativo**
> (1.0–1.1 σ) en Video_268 y Video_583. El argumento para detectar sobre `center_px` se sostiene
> igual —de hecho se refuerza—, pero la cifra no se puede citar como propiedad general.

**Paquetes necesarios**

&nbsp;&nbsp;&nbsp;&nbsp;`numpy`, `pandas`, `scipy`, `opencv-python`, `scikit-learn`, `matplotlib`, `openpyxl`

**Requisitos de archivos**

&nbsp;&nbsp;&nbsp;&nbsp;Un vídeo del gel (`.mp4` / `.avi`). El cuaderno se ejecuta desde la raíz del
proyecto (`gel_contractility/`), donde está `main.py`. No hace falta ImageJ.


---

### Mapa del flujo

| Paso | Qué entra | Qué sale |
|---|---|---|
| 0. Entorno | — | módulos cargados, rutas resueltas |
| 1. Parámetros | criterio del operador | configuración única para todo el cuaderno |
| 2. Lectura del vídeo | `.mp4` / `.avi` | metadatos, **eje temporal desde los timestamps**, mapa de máxima intensidad |
| 3. Región útil (*gauge region*) | mapa de máxima intensidad | rango `x_start`–`x_end`, bordes aproximados, alternativas descartadas, **veredicto contra el criterio de aceptación** |
| 4. Preprocesado y borde subpíxel | un fotograma + región útil | fotograma normalizado + perfil de una columna |
| 5. Ajuste robusto (RANSAC) | ~60 bordes crudos | *overlay* inlier/outlier + grosor y posición de ese fotograma |
| 6. Series temporales | vídeo completo | tabla con 4 canales por fotograma |
| 7. Elección del observable | los 4 canales | cuál tiene la señal: ruido, asimetría y SNR de cada uno |
| 8. Señal sin deriva, ruido y umbral | canal elegido | señal sin deriva (mediana móvil), signo de los eventos, ruido (MAD) |
| 9. Detección de contracciones | señal sin deriva + `k × ruido` | tabla de eventos (los del reporte) |
| **10. Elección de `k` y verificación** | serie temporal | **meseta, `k` elegido automáticamente**, control de falsos positivos |
| 11. Adelgazamiento y poblaciones | eventos + grosor | promedio de eventos alineados, cociente adelgazamiento/traslación |
| **11c. Cinética** | eventos del reporte | TTP, RT50 y amplitud relativa, con cotas cuando no son medibles |
| **12. Ritmo y enganche de fase** | eventos | estimulados vs espontáneos, período del tren, p-valor, contraste contra el estimulador |
| 13. Exportación | todo lo anterior | `serie_temporal.xlsx`, `contracciones.xlsx` (generado por el script del reporte) y los PNG numerados |

**Los tres chequeos de aceptación** (ninguno es opcional; ver `claude/protocolo-analisis-videos.md`):

1. Variación de grosor dentro de la ROI **< 6 %** (sección 3).
2. `outlier_frac` medio **< 10 %** (sección 6).
3. **Meseta** en el escaneo de umbral con **0 falsos** de control (sección 10). Sin meseta, el
   conteo de eventos **no se reporta**.

### Antes de citar un número: lo que hay que saber

- **Sin meseta no hay conteo, y tampoco si el conteo cambia con la ventana del detrend.** Desde la
  Fase 2.2 la ventana es automática (al menos 3 veces el evento más largo, mínimo 2 s) y el conteo
  se repite con 0.75×, 1× y 1.5× esa ventana: si no coincide, no se reporta. Video_491 tiene eventos
  de ~1 s; con la ventana fija de 2 s la mediana se comía la contracción y no había meseta. Con la
  ventana automática (3.1 s) da 2 eventos, estables.
- **Un evento se separa de su vecino por prominencia, no por tiempo** (Fase 2.2). La señal tiene que
  bajar entre los dos picos. Con la regla vieja (`SEP_S = 0.3` s) Video_prueba daba 28 porque borraba
  una contracción real de una ráfaga; ahora da 29.
- **TTP y RT50 sólo son medibles en Video_466 y Video_583.** En los demás la contracción dura 2–3
  fotogramas (a 30 fps) y sólo se informa una cota (`TTP < 100 ms`). Aun donde se informan, el intervalo
  de incertidumbre es ancho (±60–120 % del valor).
- **Todo está en píxeles, sin calibrar.** En Video_prueba, una medición independiente (correlación de
  intensidad de todo el perfil vertical) coincide con `center_px` en cuándo y cómo ocurre cada
  evento (correlación 0.92) pero da una amplitud varias veces menor. Comparar amplitudes entre eventos
  y videos; no citar el valor absoluto en píxeles como un desplazamiento calibrado.
- **Fotogramas `REJECTED` (NaN):** desde la Fase 2.1 no anulan el conteo (no se interpolan, no cuentan
  para el ruido). El reporte informa cuántos hubo y marca los eventos con un hueco pegado al pico.

---

## 0. Preparación del entorno

Se cargan los módulos del proyecto (`src/`) y, con `importlib`, los dos scripts de `scripts/`
que no son importables como paquete. **No se reimplementa nada**: este cuaderno llama exactamente
a las mismas funciones que `main.py` y `scripts/contraction_report.py`, sólo que deteniéndose en
cada paso para mostrar el resultado intermedio.

In [ ]:
import sys, warnings, importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Raíz del proyecto: la carpeta que contiene main.py y src/
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import io_utils, preprocessing, edge_detection, robust_fitting, pipeline
from src import qc_visualization as qc
from src import cinetica as cin
from src import plotting                      # fija el backend Agg para guardar PNG
from src.output_paths import video_output_dir


def _load_script(nombre):
    # Carga un archivo de scripts/ como módulo (no son un paquete importable).
    spec = importlib.util.spec_from_file_location(nombre, PROJECT_ROOT / "scripts" / f"{nombre}.py")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

cr = _load_script("contraction_report")   # detección sobre center_px, promedio alineado

warnings.filterwarnings("ignore")
print("Raíz del proyecto:", PROJECT_ROOT.name + "/")

## 1. Parámetros del análisis

**Todo lo ajustable vive en esta celda.** El resto del cuaderno no contiene ningún número mágico.

- **`VIDEO`** — ruta del vídeo a analizar.
- **`PX_TO_MM`** — **se deja en 1.0 por decisión del proyecto.** Los vídeos no se graban todos al
  mismo aumento, así que un factor único no tendría sentido. Todos los resultados quedan en
  píxeles y las comparaciones entre vídeos se hacen en términos relativos (porcentaje del grosor,
  cocientes) o dentro de un mismo aumento.
- **`BASE_TIEMPO`** — `"pts"` usa los timestamps que trae el contenedor; `"frames"` usa
  `fotograma / fps`. **Dejar en `"pts"`.** Ver el punto 1 de "qué cambió en la v4".
- **`FPS_OVERRIDE`** — sólo se usa si `BASE_TIEMPO="frames"`. El fps real de captura medido sobre
  los timestamps es 30.000 en los cinco vídeos analizados (el `dt` mediano da 33.333 ms),
  independientemente del fps declarado, que va de 28.97 a 29.87.
- **`half_window`** — semiancho (px) de la ventana donde se busca el borde en cada fotograma.
  Debe cubrir el rango máximo de deformación esperado en Y.
- **`min_gradient`** — gradiente mínimo para aceptar un borde. Se calibra mirando la sección 4.
  **Ojo:** se compara contra el gradiente de la imagen *ya pasada por CLAHE*.
- **`ransac_degree`** — grado del polinomio que modela el borde. **2**, no 1: el gel real tiene
  curvatura, y con una recta esa curvatura se clasifica como *outlier*.
- **`roi_min_column_spacing_px`** — separación mínima entre columnas muestreadas. El ancho mínimo
  de la ROI sale de `n_columns × este valor`, **no** de una fracción del largo del gel.
- **`roi_max_variacion_pct`** — el criterio de aceptación de la ROI. 6 % por protocolo.
- **`CANAL`** — observable sobre el que se detectan los eventos. Por defecto `"center_px"`, que es el canal del protocolo y el que usa
  el reporte. `"auto"` deja que la sección 7 lo elija por SNR, pero en Video_063 y Video_268 elige
  `y_top_px` / `y_bottom_px` y el conteo cambia (8 eventos contra 6 en Video_063): no usarlo para cifras a citar.
- **`AMP_K`** — **dejar en `None` (automático)**. Con un número se fuerza a mano, y entonces hay
  que justificar por qué se contradice al escaneo de estabilidad.
- **`WIN_DERIVA_S`** — ventana (s) de la mediana móvil que quita la deriva lenta. **Dejar en `None`
  (automática):** al menos 3 veces la duración del evento más largo, mínimo 2 s. Tiene que ser bastante
  más larga que un evento: si no, la mediana "baja con el evento" y se come la contracción (con 2 s,
  Video_491, cuyos eventos duran ~1 s, perdía la mitad). Un número la fija a mano.
- **`SEP_S`** — separación mínima (s) entre dos eventos. **Dejar en `None`:** desde la Fase 2.2 un evento
  se separa del vecino por su prominencia. Si se fija, se queda con el pico más alto de cada ventana y
  borra el otro.
- **`FREQ_ESTIMULO`** — frecuencia (Hz) configurada en el estimulador, sólo para contrastarla con la medida.
  `None` si no se sabe.


In [ ]:
%matplotlib inline

# --- Entrada ---------------------------------------------------------------
VIDEO      = PROJECT_ROOT / "data" / "raw_videos" / "Video_prueba.mp4"
PX_TO_MM   = 1.0     # fuera de alcance del proyecto: todo queda en PIXELES

# --- Base de tiempo --------------------------------------------------------
BASE_TIEMPO  = "pts"   # "pts" (timestamps del contenedor) | "frames" (fotograma/fps)
FPS_OVERRIDE = None    # solo con BASE_TIEMPO="frames"; el fps real medido es 30.000

# --- Mapa de máxima intensidad --------------------------------------------
MAXPROJ_STRIDE = 5   # procesa 1 de cada N fotogramas; 1 si el vídeo es corto

# --- Detección de bordes, ajuste robusto y gauge region --------------------
CFG = pipeline.PipelineConfig(
    n_columns                 = 60,
    half_window               = 15,
    min_gradient              = 5.0,
    edge_method               = "parabolic",   # "parabolic" | "sigmoid"

    fit_method                = "ransac",      # "ransac" | "median"
    ransac_degree             = 2,             # 2, no 1: el borde real tiene curvatura
    ransac_residual_threshold = None,          # None = adaptativo (3 x MAD del fotograma)
    ransac_residual_k         = 3.0,
    ransac_residual_floor     = 0.4,

    roi_x_start               = None,          # None = automático
    roi_x_end                 = None,
    roi_thickness_tolerance   = 0.05,
    roi_min_gradient          = 10.0,
    roi_max_slope             = 0.02,
    roi_min_column_spacing_px = 3.0,           # ancho minimo ROI = n_columns * esto
    roi_max_variacion_pct     = 6.0,           # criterio de aceptación del protocolo

    base_tiempo               = BASE_TIEMPO,
    fps_override              = FPS_OVERRIDE,

    px_to_mm                  = PX_TO_MM,
    use_clahe                 = True,
    use_denoise               = False,
    savgol_window             = 11,
    savgol_polyorder          = 3,
    low_quality_frac          = 0.30,
)

# --- Elección del observable y detección de eventos ------------------------
CANAL         = "center_px"   # por protocolo (CLAUDE.md): se detecta sobre center_px. "auto" elige por SNR, pero puede elegir y_top/y_bottom
AMP_K         = None     # None = lo elige la meseta (sección 10). Un número lo fuerza.
WIN_DERIVA_S  = None     # None = automática: >= 3 x el evento más largo, mínimo 2 s
SEP_S         = None     # None = eventos separados por prominencia (Fase 2.2)
HALF_EVENTO_S = 1.5      # semiventana del promedio de eventos alineados (sección 11)
FREQ_ESTIMULO = 0.1      # Hz configurados en el estimulador; None si no se sabe

# --- Control de calidad -----------------------------------------------------
FRAME_QC = 0             # fotograma sobre el que se hace la inspección visual

# --- Salida -----------------------------------------------------------------
CALIBRADO = abs(PX_TO_MM - 1.0) > 1e-9
UNIDAD    = "mm" if CALIBRADO else "px (SIN CALIBRAR)"
OUT_DIR   = video_output_dir(VIDEO)
print("Vídeo      :", VIDEO.name)
print("Salida     :", OUT_DIR.relative_to(PROJECT_ROOT))
print("Unidad     :", UNIDAD)
print("Base tiempo:", BASE_TIEMPO)


## 2. Lectura del vídeo, base de tiempo y mapa de máxima intensidad

**Entra:** el vídeo crudo. **Sale:** sus metadatos, **el eje temporal** y una imagen resumen.

El vídeo se lee fotograma a fotograma con un generador (`io_utils.frame_generator`), nunca entero
en memoria.

### La base de tiempo (esto cambió en la v4)

`tiempo = fotograma / fps` supone que no falta ningún fotograma. **El `fps` que declara el
archivo es el promedio** `(n−1)/duración`, así que si la grabación perdió fotogramas ese promedio
ya baja y el eje se come los huecos: los eventos aparecen más juntos de lo que ocurrieron.

El contenedor trae un timestamp por fotograma. El espaciado **típico** entre fotogramas
consecutivos sí es el período real de captura: da 33.333 ms (= 30.0003 fps) en los cinco vídeos
medidos, con fps declarados que van de 28.97 a 29.87. Los huecos son los `dt` que valen un
múltiplo entero de esa mediana.

| vídeo | fps declarado | fotogramas perdidos |
|---|---|---|
| Video_063 | 29.870 | 26 (1.39 %) |
| Video_268 | 29.612 | 43 (1.85 %) |
| **Video_466** | **28.966** | **103 (4.73 %)** |
| Video_491 | 29.661 | 28 (1.49 %) |
| Video_583 | 29.674 | 33 (1.45 %) |

Dos trampas técnicas, por si hay que reimplementarlo:

- `ffprobe -show_entries packet=pts_time` devuelve los timestamps en **orden de decodificación**;
  con B-frames no son monótonos y la mitad de los deltas salen negativos. Hay que ordenarlos.
- `cv2.CAP_PROP_POS_MSEC` leído **antes** de `read()` devuelve el timestamp del fotograma
  *anterior*. Hay que leerlo después.

### El mapa de máxima intensidad

Guarda, para cada píxel, el valor máximo que alcanzó a lo largo del vídeo. Es el equivalente de
*Z-Project > Max Intensity* de ImageJ. Sirve para saber **dónde puede estar el borde del gel en
cualquier instante** sin indicarlo a mano.

**Ejes de la figura:** `x` = columna de píxeles (0 a ancho−1); `y` = fila de píxeles (0 arriba,
creciente hacia abajo — convención de imagen). El gris es intensidad máxima acumulada.


In [ ]:
meta = io_utils.get_video_metadata(str(VIDEO))

# Timestamps reales del contenedor: son los que detectan los fotogramas perdidos.
pts = io_utils.read_pts_seconds(str(VIDEO))
dts = np.diff(np.sort(pts))
dt_med  = float(np.median(dts))
fps_pts = 1.0 / dt_med
huecos  = dts[dts > 1.5 * dt_med]
faltan  = float(np.round((huecos / dt_med - 1).sum()))

# fps DECLARADO por el archivo (promedio): sólo para mostrar. Todo el análisis usa el fps de los
# timestamps (sección 6).
fps_declarado = FPS_OVERRIDE if FPS_OVERRIDE else (meta["fps"] if meta["fps"] > 0 else 30.0)
meta["fps declarado"]        = round(fps_declarado, 4)
meta["duracion_s"]           = round(meta["n_frames"] / fps_declarado, 2)
meta["fps segun PTS"]        = round(fps_pts, 4)
meta["huecos en PTS"]        = int(len(huecos))
meta["fotogramas perdidos"]  = int(faltan)
meta["perdidos (%)"]         = round(100 * faltan / (len(pts) + faltan), 2)
display(pd.DataFrame([meta]).T.rename(columns={0: "valor"}))

if faltan / (len(pts) + faltan) > 0.01:
    print(f"AVISO: la grabación perdió ~{faltan:.0f} fotogramas ({100*faltan/(len(pts)+faltan):.1f} %), "
          f"repartidos en {len(huecos)} huecos.")
    if BASE_TIEMPO != "pts":
        print("       Con el eje fotograma/fps los huecos se comen y los eventos parecen MAS")
        print("       JUNTOS de lo que fueron. Poner BASE_TIEMPO = 'pts'.")

max_proj = io_utils.compute_max_projection(str(VIDEO), stride=MAXPROJ_STRIDE)

fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(max_proj, cmap="gray")
ax.set_title(f"Mapa de máxima intensidad — {VIDEO.name} (1 de cada {MAXPROJ_STRIDE} fotogramas)")
ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
plt.show()


## 3. Región útil de medición (*gauge region*)

**Entra:** el mapa de máxima intensidad. **Sale:** el rango de columnas donde se va a medir, la
posición aproximada de los bordes, y **el veredicto contra el criterio de aceptación**.

El gel tiene forma de reloj de arena: delgado y uniforme en el centro, más ancho cerca de los
anclajes. Medir junto a un anclaje es incorrecto desde lo experimental, porque ahí manda la
concentración de tensiones del anclaje y no la contractilidad del material.

`preprocessing.auto_detect_roi` sigue la franja columna a columna desde el centro hacia afuera,
mide grosor y nitidez de cada columna, estima la **cintura** del gel, y busca el bloque contiguo
más largo que cumpla, en cascada de menos a más permisivo:

| nivel | criterio |
|---|---|
| `gauge_plana` | nítida + cerca de la cintura + perfil plano |
| `gauge_cintura` | nítida + cerca de la cintura |
| `gauge_relajada` | nítida + tolerancia de grosor al doble |
| `solo_nitidez` | sólo nitidez |
| `franja_completa` | todo lo que se pudo seguir |
| **`gauge_rescate_plana`** | **(v4) sólo si la cascada no dio nada plano: barrido de la ventana más ancha con variación ≤ 6 %** |

### Los dos cambios de la v4 acá

**El ancho mínimo ya no es una fracción del gel.** Era `0.35 × (columnas con gel)`, un valor
fijado mirando Video_063. Con geles que ocupan ~1850 columnas eso exige una ROI de ~650 px, y las
*gauge regions* reales de la batería miden 225–470 px: ningún nivel estricto podía cumplirlo y la
cascada caía sola hasta `solo_nitidez` o `franja_completa`. Video_466 terminó tomando el ancho
entero del cuadro, con **164 % de variación de grosor adentro** y un residuo de ajuste de 7.0 px
contra los 0.8 px normales.

Ahora el mínimo sale de cuántas columnas se muestrean: `n_columns × roi_min_column_spacing_px`.
No depende de cuán largo es el gel, sino de cuántas columnas se quieren medir y de cuán juntas
pueden estar.

> **Ojo con el "3 px":** se eligió para que columnas vecinas no compartan el mismo ruido de imagen,
> pero eso **no se midió** (la revisión de código lo anotó como H19). Tampoco se sostiene la idea de
> que separarlas evite compartir el tile de CLAHE: con la grilla de 8 × 8 de CLAHE, en un cuadro de
> 1920 × 1080 cada tile mide 240 × 135 px, y una ROI de 180–240 px cae en 1 o 2 tiles sea cual sea la
> separación. Es un valor práctico, no derivado. En Video_466 los 180 px resultantes rechazaron la
> cintura real (150 px) y el rescate eligió otra ventana.

**El rescate va DESPUÉS de la cascada, no antes.** La cascada evalúa el *bloque contiguo más
largo* de cada criterio; si el gel tiene una cintura corta dentro de una franja larga, ese bloque
incluye los hombros. El rescate busca por barrido la ventana más ancha que sí es plana.

Se probó primero como nivel prioritario y **empeoraba los dos vídeos validados**: maximizar el
ancho contra el límite del 6 % se pega al borde y agarra columnas del hombro.

| | cascada (`gauge_cintura`) | barrido |
|---|---|---|
| Video_prueba | 5.32 %, residuo 0.787 px | 5.98 %, residuo 0.817 px |
| Video_063 | 4.91 %, residuo 0.822 px | 5.96 %, residuo 1.052 px |

Es una red de seguridad, no una mejora. Con la cascada corregida, tres de los cuatro vídeos de la
batería se resuelven solos y el rescate sólo se activa en Video_466.

> **Chequeo de aceptación 1:** variación de grosor dentro de la ROI **< 6 %**. La hoja `resumen`
> lo registra como `ROI cumple criterio`. Con `--exigir-roi`, `main.py` aborta en vez de seguir
> emitiendo números.

**Ejes de `00_roi_profile.png`:** panel superior — `x` = columna de la imagen (px), `y` = grosor
de la franja en esa columna (px); la punteada es la cintura y la banda verde la ROI elegida.
Panel inferior — mismo `x`, `y` = nitidez del borde, definida como el **mínimo** del gradiente
vertical máximo de los dos bordes (mínimo, no máximo: los dos tienen que ser nítidos).


In [ ]:
roi = preprocessing.auto_detect_roi(
    max_proj,
    thickness_tolerance  = CFG.roi_thickness_tolerance,
    min_gradient_for_roi = CFG.roi_min_gradient,
    max_thickness_slope  = CFG.roi_max_slope,
    x_start              = CFG.roi_x_start,
    x_end                = CFG.roi_x_end,
    n_columns            = CFG.n_columns,
    min_column_spacing_px= CFG.roi_min_column_spacing_px,
    max_variacion_pct    = CFG.roi_max_variacion_pct,
)
x_positions = np.linspace(roi["x_start"], roi["x_end"] - 1, CFG.n_columns).astype(int)

q = roi["roi_quality"]
display(pd.DataFrame([{k: v for k, v in q.items() if k != "alternativas"}]).T
        .rename(columns={0: "valor"}))

if q.get("alternativas"):
    print("Alternativas de ROI consideradas (la elegida es la primera suficientemente ancha):")
    display(pd.DataFrame(q["alternativas"]))

print("CHEQUEO 1 - variacion de grosor en la ROI:",
      f"{q.get('variacion_en_roi_pct')}%  ->",
      "OK" if q.get("cumple_criterio_aceptacion") else "FALLA (mirar el perfil y forzar --x-start/--x-end)")
print(f"Columnas muestreadas: {CFG.n_columns} entre x={roi['x_start']} y x={roi['x_end']} "
      f"(1 cada {(roi['x_end']-roi['x_start'])/CFG.n_columns:.1f} px)")

# Perfil de grosor y nitidez con la ROI marcada
qc.plot_roi_profile(roi, OUT_DIR / "00_roi_profile.png")
display(Image(str(OUT_DIR / "00_roi_profile.png")))

# La misma ROI, sobre la imagen
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(max_proj, cmap="gray")
xs = np.arange(max_proj.shape[1])
ax.plot(xs, roi["top_guess"],    color="#56C48F", lw=1, label="borde superior aprox.")
ax.plot(xs, roi["bottom_guess"], color="#56C48F", lw=1)
ax.axvspan(roi["x_start"], roi["x_end"], color="#FFCA7F", alpha=0.25, label="gauge region")
ax.vlines(x_positions, roi["top_guess"][x_positions] - 20,
          roi["bottom_guess"][x_positions] + 20, color="#FFCA7F", lw=0.4, alpha=0.8)
ax.set_title("Región útil de medición y columnas muestreadas")
ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)"); ax.legend(loc="upper right", fontsize=8)
plt.show()


## 4. Preprocesado y localización subpíxel del borde

**Entra:** un fotograma y la región útil. **Sale:** el fotograma normalizado y el perfil de
intensidad/gradiente de una columna, con el borde localizado.

CLAHE normaliza el contraste **localmente** (por celdas), lo que compensa el viñeteo y las
variaciones de iluminación entre fotogramas. Comprobado sobre Video_063: con CLAHE el ruido de
`center_px` baja de 0.0495 a 0.0338 px (1.5× mejor) y la amplitud medida de los eventos cambia
sólo un 2 % — o sea que **mejora el SNR sin deformar la señal**.

Sobre cada columna se calcula el gradiente de intensidad y se ajusta una parábola a los tres
puntos alrededor de su máximo: el vértice de esa parábola es la posición del borde **con
resolución de fracciones de píxel**. Si el gradiente máximo no supera `min_gradient`, la columna
se descarta en vez de forzar una medición dudosa.

**Ejes de la figura de perfil:** `x` = fila `y` de la imagen (px) dentro de la ventana de
búsqueda; una curva es la intensidad del píxel (0–255) y la otra el gradiente vertical
(unidades de intensidad por píxel). La vertical marca el vértice de la parábola, es decir el
borde subpíxel.

In [ ]:
idx0, frame0 = next(io_utils.frame_generator(str(VIDEO), start_frame=FRAME_QC))
frame_p = preprocessing.preprocess_frame(frame0, use_clahe=CFG.use_clahe,
                                         use_denoise=CFG.use_denoise)

x0, x1 = roi["x_start"], roi["x_end"]
y0 = int(max(0, roi["top_guess"][x0:x1].min() - 40))
y1 = int(min(frame0.shape[0], roi["bottom_guess"][x0:x1].max() + 40))

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].imshow(frame0[y0:y1, x0:x1], cmap="gray"); axes[0].set_title(f"Fotograma {idx0} — crudo")
axes[1].imshow(frame_p[y0:y1, x0:x1], cmap="gray")
axes[1].set_title("Tras CLAHE (lo que analiza el algoritmo)")
for a in axes: a.set_yticks([])
plt.tight_layout(); plt.show()

# Perfil de UNA columna: intensidad, gradiente y borde subpíxel
x_col = int(np.median(x_positions))
fig = qc.plot_column_profile(frame_p, x_col, roi["top_guess"][x_col],
                             half_window=CFG.half_window, polarity=+1,
                             min_gradient=CFG.min_gradient)
plt.show()

pt = edge_detection.subpixel_edge_parabolic(
    frame_p[:, x_col].astype(float),
    int(roi["top_guess"][x_col] - CFG.half_window),
    int(roi["top_guess"][x_col] + CFG.half_window),
    polarity=+1, min_gradient=CFG.min_gradient)
print(f"Columna x={x_col}: borde superior en y={pt.y:.3f} px "
      f"(gradiente={pt.quality:.2f}, válido={pt.valid})")

## 5. Ajuste robusto: qué columnas se aceptan y cuáles se descartan

**Entra:** los ~60 bordes crudos del fotograma. **Sale:** el modelo ajustado, la lista de columnas
descartadas, y el grosor **y la posición** de ese fotograma.

Con 60 columnas es esperable que alguna esté tapada por una burbuja. Promediarlas todas diluiría
ese error. RANSAC ajusta el modelo a subconjuntos aleatorios, se queda con el que reúne más
*inliers* y **excluye** el resto por completo.

Dos cambios importantes respecto de la versión anterior:

- **`ransac_degree = 2`.** Con grado 1 (recta), la curvatura real del gel se clasificaba como
  outlier: en un fotograma limpio aparecían 9 outliers **contiguos** en el borde superior. Los
  outliers contiguos indican que el modelo está mal, los dispersos indican burbujas.
- **Umbral adaptativo.** En vez de 1.5 px fijo, se usa 3× el MAD de los residuos del propio
  fotograma. Con el umbral fijo, ~70 % de la varianza de la serie de grosor era artefacto del
  ajuste, no movimiento del gel.

En el *overlay*: verde = borde aceptado, rojo = borde detectado pero descartado como outlier,
amarillo = columna sin borde confiable, cian = modelo final (lo que realmente define el grosor).

In [ ]:
diag = qc.run_frame_diagnostics(frame_p, x_positions,
                                roi["top_guess"], roi["bottom_guess"], CFG)
overlay = qc.draw_diagnostics_overlay(frame_p, diag, px_to_mm=PX_TO_MM)

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.imshow(overlay[y0:y1, x0:x1, ::-1])
ax.set_title(f"Fotograma {idx0} — verde: inlier · rojo: outlier · amarillo: sin borde · cian: ajuste")
ax.set_xticks([]); ax.set_yticks([])
plt.show()

tabla = qc.diagnostics_to_dataframe(diag)
n = len(tabla)
print(f"Grosor del fotograma {idx0}: {diag.thickness_px:.3f} px"
      + (f"  ({diag.thickness_px * PX_TO_MM:.4f} mm)" if CALIBRADO else ""))
print(f"Columnas: {n} | borde superior outlier: {int((~tabla.top_is_inlier).sum())}"
      f" | borde inferior outlier: {int((~tabla.bottom_is_inlier).sum())}"
      f" | sin borde: {int(tabla.y_top_raw.isna().sum() + tabla.y_bottom_raw.isna().sum())}")

# Outliers CONTIGUOS = el modelo está mal.  DISPERSOS = burbujas.
for lado, col in (("superior", "top_is_inlier"), ("inferior", "bottom_is_inlier")):
    malos = np.flatnonzero(~tabla[col].to_numpy())
    if len(malos) >= 3:
        bloques = np.split(malos, np.flatnonzero(np.diff(malos) > 1) + 1)
        mayor = max(len(b) for b in bloques)
        diagn = ("CONTIGUOS -> el modelo no representa la geometría: subir ransac_degree"
                 if mayor >= 3 else "dispersos -> compatible con burbujas/motas")
        print(f"  borde {lado}: {len(malos)} outliers, bloque mayor de {mayor} -> {diagn}")

display(tabla.head(8).round(3))

## 6. Series temporales (vídeo completo)

**Entra:** el vídeo entero. **Sale:** una fila por fotograma, con **cuatro canales** en vez de uno.

Se repite el ciclo de las secciones 4 y 5 en cada fotograma. Cada uno queda etiquetado como `OK`,
`LOW_QUALITY` (más de `low_quality_frac` de las columnas descartadas, contando los dos bordes) o
`REJECTED` (ningún ajuste confiable); los `REJECTED` quedan como `NaN`, no se inventan.

Los cuatro canales, y qué ve cada uno:

| columna | qué es | ciego a |
|---|---|---|
| `y_top_px` | posición del borde superior (px, y crece hacia abajo) | — |
| `y_bottom_px` | posición del borde inferior | — |
| `thickness_px` | `y_bottom − y_top` | **traslación**: si los dos bordes bajan 1 px, no cambia |
| `center_px` | `(y_bottom + y_top) / 2` | **cambio de grosor**: si el borde de arriba baja 1 px y el de abajo sube 1 px, no cambia |

`thickness_px` y `center_px` son **complementarios**: entre los dos describen todo el movimiento
vertical de la franja. Guardar sólo uno es tirar la mitad de la información, que es exactamente lo
que hacía la versión anterior de este cuaderno.

Además se guardan `residual_top_px` y `residual_bottom_px` (MAD de los residuos del ajuste, por
fotograma). **Si esa columna se mueve junto con el grosor, lo que se está midiendo es el ajuste,
no el gel** — es el control interno del método.

Al final se aplica un suavizado temporal **Savitzky-Golay** en lugar de una media móvil: la media
móvil aplanaría los picos de contracción, mientras que Savitzky-Golay ajusta un polinomio local y
reduce el ruido **preservando la forma** de los eventos rápidos.

> Esta celda recorre todo el vídeo; es la más lenta del cuaderno.

In [ ]:
import cv2
OUT_DIR.mkdir(parents=True, exist_ok=True)
maxproj_path = OUT_DIR / "00_max_projection.png"
cv2.imwrite(str(maxproj_path), max_proj)

df = pipeline.process_video(str(VIDEO), str(maxproj_path), CFG)
t  = df.time_s.to_numpy()

# fps efectivo de la serie: el MISMO que calcula contraction_report. Con BASE_TIEMPO="pts" sale de los
# timestamps del contenedor (30.000), no del fps declarado por el archivo (29.x).
fps = 1.0 / float(np.median(np.diff(t)))
print(f"fps de la serie: {fps:.4f}")

resumen = {
    "fotogramas totales"     : len(df),
    "OK"                     : int((df.frame_quality == "OK").sum()),
    "LOW_QUALITY"            : int((df.frame_quality == "LOW_QUALITY").sum()),
    "REJECTED"               : int((df.frame_quality == "REJECTED").sum()),
    "grosor medio (px)"      : round(float(df.thickness_px.mean()), 3),
    "recorrido grosor (px)"  : round(float(df.thickness_px.max() - df.thickness_px.min()), 3),
    "recorrido centro (px)"  : round(float(df.center_px.max() - df.center_px.min()), 3),
    "outlier_frac medio"     : round(float(df.outlier_frac.mean()), 4),
    "residuo sup medio (px)" : round(float(df.residual_top_px.mean()), 4),
    "residuo inf medio (px)" : round(float(df.residual_bottom_px.mean()), 4),
}
display(pd.DataFrame([resumen]).T.rename(columns={0: "valor"}))
display(df.head(5).round(4))

n_rechazados = int((df.frame_quality == "REJECTED").sum())
if n_rechazados:
    print(f"AVISO: {n_rechazados} fotograma(s) REJECTED (NaN). Con un solo NaN el reporte calcula ruido NaN y da "
          f"0 eventos sin avisar. Revisar la sección 5 o reprocesar con otra ROI antes de seguir.")

if df.outlier_frac.mean() > 0.10:
    print("AVISO: se descarta más del 10 % de las columnas en promedio. Unos pocos por ciento es "
          "normal con burbujas; 10 % sostenido, no. Revisar la sección 5.")

# Los cuatro canales, uno debajo del otro y en la misma escala de tiempo
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
for ax, c, col in zip(axes, ["thickness_px", "center_px", "y_top_px", "y_bottom_px"],
                      ["crimson", "steelblue", "darkgreen", "darkorange"]):
    ax.plot(t, df[c], color=col, lw=0.7)
    ax.set_ylabel(f"{c}\n({UNIDAD})", fontsize=8)
    ax.grid(alpha=0.3)
axes[0].set_title("Los cuatro canales — mismo vídeo, misma escala de tiempo, distinta información")
axes[-1].set_xlabel("Tiempo (s)")
plt.tight_layout(); plt.show()

display(Image(str(plotting.plot_timeseries(df, OUT_DIR, unit_label=UNIDAD, calibrated=CALIBRADO))))

## 7. Elección del observable — ¿en qué canal está la contracción?

**Entra:** los cuatro canales. **Sale:** cuál de ellos se usa para detectar, y por qué.

Ésta es la sección nueva, y es la que cambia el resultado del análisis.

La intuición de que una contracción tiene que verse como **adelgazamiento** supone que los dos
extremos del gel están perfectamente fijos y que la contracción es simétrica. En la práctica, en
este montaje, la mayor parte del movimiento es un **desplazamiento vertical de toda la franja**.

**Cuánto del movimiento es cambio de grosor depende del vídeo**, y eso se supo recién con la
batería. Medido sobre el promedio de eventos alineados (la medida robusta de la sección 11):

| vídeo | adelgazamiento robusto | significancia | % de la traslación |
|---|---|---|---|
| Video_prueba | 0.577 px | 57 σ | 18.2 % |
| Video_063 | 0.192 px | 11 σ | 12.8 % |
| **Video_268** | **−0.037 px** | **no significativo** | **−2.4 %** |
| **Video_583** | **0.030 px** | **1.1 σ, no significativo** | **1.0 %** |

En Video_268 y Video_583 el grosor **no se mueve de forma medible**: la contracción es traslación
casi pura. Eso refuerza el argumento para detectar sobre `center_px` —ahí no es una preferencia,
es la única opción— pero **el "13–19 %" no se puede citar como una constante del montaje**. Es un
número por vídeo.

Para decidir sin suponer nada, se comparan los cuatro canales con tres números, todos calculados
después de quitar la deriva lenta con una **mediana móvil** (mediana, no pasabanda: un pasabanda
convierte cada evento en un valle flanqueado por dos picos falsos y destruye la asimetría que
justamente se quiere medir):

- **`ruido_MAD_px`** — mediana de las desviaciones absolutas, escalada para que sea comparable a
  un desvío estándar. Es el piso de ruido del canal.
- **`skew`** — asimetría. Una contracción es una excursión en **un solo sentido**, y el gel pasa
  más tiempo relajado que contraído, así que la distribución queda con una cola larga hacia ese
  lado. El ruido de medición es simétrico: `skew ≈ 0`. Este criterio **no depende de ningún
  umbral**.
- **`max_excursion_MAD`** — la mayor excursión medida en unidades de ruido. Es, literalmente, el
  SNR del canal.

`center_px` tiene un SNR muy superior al de `thickness_px` en todos los vídeos medidos:

| vídeo | SNR `center_px` | SNR `thickness_px` |
|---|---|---|
| Video_prueba | 82 | 17 |
| Video_063 | 46 | 9 |
| Video_268 | 33 | 10 |
| Video_466 | 21 | 5 |
| Video_583 | 44 | 10 |
| Video_491 | 12 | 6 |

**Pero `center_px` no siempre es el canal de mayor SNR**: en Video_063 y Video_268, `y_top_px` o
`y_bottom_px` superan algo a `center_px`. Como el protocolo fija `center_px` y el reporte detecta
sobre ese canal, el cuaderno también (`CANAL = "center_px"`). La tabla sigue mostrando los cuatro
canales; si el de mayor SNR es otro, la celda de abajo lo avisa.

**Ejes de la figura:** panel izquierdo — `x` = tiempo (s), `y` = el canal después de quitarle la
deriva (px); las líneas grises punteadas están a ±4×MAD. Panel derecho — histograma del mismo
canal, `x` = valor (px), `y` = número de fotogramas **en escala logarítmica** (log, porque lo
interesante son las colas, que en escala lineal no se ven).


In [ ]:
from scipy.stats import skew as _skew

CANALES = ["thickness_px", "center_px", "y_top_px", "y_bottom_px"]

# Ventana del detrend con la MISMA regla que el reporte (Fase 2.2): >= 3 x el evento más largo, mínimo 2 s.
_vent = cr.ventana_deriva(df["center_px"].to_numpy(float), fps)
WIN_USADO = WIN_DERIVA_S if WIN_DERIVA_S else _vent["win_s"]
print(f"Ventana del detrend: {WIN_USADO:g} s "
      f"({'automática' if not WIN_DERIVA_S else 'fijada a mano'}; evento claro más largo: "
      f"{_vent['duracion_max_s']:.2f} s)")

filas, sin_deriva = [], {}
for c in CANALES:
    r = cr.detrend_median(df[c].to_numpy(float), fps, WIN_USADO)
    m = cr.mad(r)
    sin_deriva[c] = (r, m)
    filas.append({
        "canal"             : c,
        "recorrido_px"      : round(float(df[c].max() - df[c].min()), 3),
        "ruido_MAD_px"      : round(m, 4),
        "skew"              : round(float(_skew(r)), 2),
        "max_excursion_MAD" : round(float(np.abs(r).max() / m), 1),
        "pct_fuera_4sigma"  : round(100 * float(np.mean(np.abs(r) > 4 * m)), 2),
    })
comparacion = pd.DataFrame(filas)
display(comparacion)

CANAL_USADO = (comparacion.loc[comparacion.max_excursion_MAD.idxmax(), "canal"]
               if CANAL == "auto" else CANAL)
print(f"Canal de detección: {CANAL_USADO}"
      + ("   (elegido automáticamente por SNR)" if CANAL == "auto" else "   (forzado a mano)"))

razon = (comparacion.set_index("canal").max_excursion_MAD[CANAL_USADO]
         / comparacion.set_index("canal").max_excursion_MAD["thickness_px"])
_mejor = comparacion.loc[comparacion.max_excursion_MAD.idxmax(), "canal"]
if _mejor != CANAL_USADO:
    print(f"Nota: el canal de mayor SNR es {_mejor}, pero se usa {CANAL_USADO} (protocolo del proyecto).")
if CANAL_USADO != "thickness_px":
    print(f"Es {razon:.1f} veces más sensible que thickness_px. El grosor no se descarta: "
          f"se mide en la sección 11, promediando eventos.")

fig, axes = plt.subplots(len(CANALES), 2, figsize=(13, 2.4 * len(CANALES)),
                         gridspec_kw={"width_ratios": [3, 1]})
for (ax1, ax2), c in zip(axes, CANALES):
    r, m = sin_deriva[c]
    ax1.plot(t, r, color="crimson" if c == CANAL_USADO else "gray", lw=0.6)
    ax1.axhline( 4 * m, color="gray", ls="--", lw=0.8)
    ax1.axhline(-4 * m, color="gray", ls="--", lw=0.8)
    ax1.set_ylabel(f"{c}\n(sin deriva, px)", fontsize=8)
    ax1.set_title(f"{c}   ruido {m:.4f} px · skew {float(_skew(r)):+.2f} · SNR "
                  f"{np.abs(r).max()/m:.0f}" + ("   <-- USADO" if c == CANAL_USADO else ""),
                  fontsize=9)
    ax1.grid(alpha=0.3)
    ax2.hist(r, bins=80, color="steelblue"); ax2.set_yscale("log")
    ax2.axvline(0, color="black", lw=0.8)
    ax2.set_ylabel("nº fotogramas (log)", fontsize=7)
axes[-1][0].set_xlabel("Tiempo (s)"); axes[-1][1].set_xlabel("valor (px)")
plt.tight_layout(); plt.show()

## 8. Señal sin deriva, ruido y umbral

**Entra:** el canal elegido en la sección 7. **Sale:** la señal sin deriva, con los eventos hacia arriba, y su ruido.

Esta sección y las dos siguientes usan **exactamente las funciones de `scripts/contraction_report.py`**,
el script que produce `contracciones.xlsx`. (Las versiones anteriores de este cuaderno usaban otro
detector, `src/event_detection.py` (borrado el 2026-10-01), que daba conteos distintos en algunos vídeos —29 en vez de 28 en
Video_prueba, 6 en vez de 5 en Video_466—; ya no se usa acá.)

Tres pasos:

1. **Quitar la deriva.** Se resta una **mediana móvil** de `WIN_USADO` segundos (automática: al menos
   3 veces el evento más largo, mínimo 2 s). Se usa mediana y
   no un filtro pasabanda: el pasabanda convierte cada evento real en un valle flanqueado por dos picos
   falsos y destruye la asimetría que se quiere medir.
2. **Dar vuelta el signo si hace falta**, para que los eventos queden hacia arriba. El signo se decide
   mirando qué cola de la distribución es más pesada, no suponiéndolo: depende de la convención de
   la imagen y de si el gel sube o baja al contraerse.
3. **Medir el ruido con MAD** (desviación absoluta mediana): `1.4826 × mediana(|x − mediana(x)|)`. Es
   comparable a un desvío estándar, pero no se distorsiona por los propios eventos grandes.

El umbral de detección es `k × ruido`. **`k` no se fija a ojo**: lo elige el escaneo de la sección 10.

> Desde la Fase 2.1, un fotograma `REJECTED` (NaN) no anula nada: no cuenta para el ruido ni puede
> ser un pico, y no se interpola.

**Ejes:** panel superior — `x` = tiempo (s), `y` = el canal de detección (px); gris, la señal cruda;
azul punteado, la deriva (la mediana móvil). Panel inferior — `x` = tiempo (s), `y` = la señal sin
deriva (px); la banda gris es ±1 ruido.


In [ ]:
# Señal sin deriva y signo: lo mismo que hace cr.analizar por dentro.
r_sel, m_sel = sin_deriva[CANAL_USADO]
SIGNO  = cr._signo_evento(r_sel)
r_det  = SIGNO * r_sel                 # ahora los eventos van hacia ARRIBA
RUIDO  = cr.mad(r_det)
deriva = df[CANAL_USADO].to_numpy(float) - r_sel

print(f"Los eventos en {CANAL_USADO} son excursiones hacia "
      f"{'ARRIBA' if SIGNO > 0 else 'ABAJO'} en coordenadas de imagen "
      f"(se multiplica por {SIGNO:+d} para que queden hacia arriba)")
print(f"Ruido (MAD) de la señal sin deriva: {RUIDO:.4f} px")
print(f"Excursión máxima: {r_det.max():.3f} px = {r_det.max()/RUIDO:.0f} veces el ruido")

fig, (a1, a2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
a1.plot(t, df[CANAL_USADO], color="lightgray", lw=0.8, label=f"{CANAL_USADO} (crudo)")
a1.plot(t, deriva, color="steelblue", lw=1.0, ls="--", label=f"deriva (mediana móvil de {WIN_USADO:g} s)")
a1.set_ylabel(f"{CANAL_USADO}\n({UNIDAD})"); a1.legend(fontsize=8)
a1.set_title(f"Canal de detección y deriva — {CANAL_USADO}")
a2.plot(t, r_det, color="crimson", lw=0.7)
a2.axhspan(-RUIDO, RUIDO, color="gray", alpha=0.25, label="±1 ruido (MAD)")
a2.axhline(0, color="black", lw=0.6)
a2.set_xlabel("Tiempo (s)"); a2.set_ylabel(f"sin deriva ({UNIDAD})")
a2.legend(fontsize=8); a2.set_title("Señal sin deriva, con los eventos hacia arriba")
plt.tight_layout(); plt.show()


## 9. Detección de contracciones

**Entra:** la señal sin deriva y el ruido. **Sale:** la tabla de eventos, **los mismos que aparecen en
`contracciones.xlsx`**.

Un evento es un pico de la señal sin deriva que cumple dos condiciones (Fase 2.2):

- **altura** ≥ `k × ruido` sobre el reposo, y
- **prominencia** ≥ `k × ruido`: tiene que sobresalir eso por encima del valle que lo separa de su vecino
  más alto. Dos contracciones de una ráfaga son dos eventos si la señal baja entre ellas; un repunte de
  ruido en la cola de un evento lento no lo es.

`find_peaks(height = k·ruido, prominence = k·ruido)`, **sin separación mínima en tiempo**. La regla vieja
(`distance = SEP_S·fps`, 0.3 s) fallaba de las dos maneras: en Video_prueba borraba una contracción real
de una ráfaga (28 en vez de 29) y en eventos lentos contaba la cola como otro evento. Detalle y
comparación en `claude/propuesta-fase-2-2.md`.

- **`k` lo elige la meseta del escaneo de la sección 10**, salvo que fuerces `AMP_K`.
- **No hay filtro de "agudeza".** Las versiones anteriores descartaban eventos cuya amplitud no bajaba al
  suavizarlos; en los seis vídeos vigentes ese filtro no descartaba ninguno.

**Columnas de la tabla de eventos:**

| columna | qué es |
|---|---|
| `evento` | número correlativo |
| `frame_idx` | fotograma del pico |
| `tiempo_s` | instante del pico (s, de los timestamps) |
| `amplitud_px` | altura del pico en la señal sin deriva (px del canal de detección) |
| `intervalo_s` | tiempo desde el evento anterior |

**Ejes de la figura:** `x` = tiempo (s), `y` = la señal sin deriva (px); la línea roja punteada es el
umbral `k × ruido` y los triángulos son los eventos.


In [ ]:
# Detección con la MISMA función que usa scripts/contraction_report.py: los eventos, el k, la
# separación de estimuladas y la cinética salen de acá y se reutilizan en las secciones 10 a 12.
a_rep = cr.analizar(df, CANAL_USADO, AMP_K, WIN_DERIVA_S, SEP_S, HALF_EVENTO_S,
                    separar=True, min_frames_cinetica=cin.MIN_FRAMES)
K_USADO = a_rep["k_usado"]
pk = a_rep["_picos"]

eventos = pd.DataFrame({
    "evento"     : np.arange(1, len(pk) + 1),
    "frame_idx"  : pk,
    "tiempo_s"   : a_rep["_t"][pk],
    "amplitud_px": a_rep["_r"][pk],
})
eventos["intervalo_s"] = eventos.tiempo_s.diff()

print(f"{len(eventos)} contracciones detectadas sobre {CANAL_USADO}   "
      f"(k = {K_USADO:g}, {'automático' if a_rep['k_automatico'] else 'forzado a mano'}; "
      f"umbral = {K_USADO * a_rep['ruido_canal_px']:.4f} px)")
print(f"Ventana del detrend: {a_rep['win_s_usado']:g} s. Conteo con otras ventanas: {a_rep['conteo_por_ventana']}")
if not a_rep["conteo_reportable"]:
    print(f">>> NO REPORTABLE: {a_rep.get('motivo_no_reportable', '')}. Los eventos sirven para auditar.")

if len(eventos):
    display(eventos.head(12).round(3))
    display(eventos[["amplitud_px", "intervalo_s"]].describe().round(3))

    fig, ax = plt.subplots(figsize=(12, 3.6))
    ax.plot(a_rep["_t"], a_rep["_r"], color="#1f77b4", lw=0.7, label=f"{CANAL_USADO} sin deriva")
    ax.axhline(K_USADO * a_rep["ruido_canal_px"], color="crimson", ls=":", lw=1.2,
               label=f"umbral = {K_USADO:g} × ruido")
    ax.plot(eventos.tiempo_s, eventos.amplitud_px, "v", color="crimson", ms=7, label=f"{len(eventos)} eventos")
    ax.axhline(0, color="gray", lw=0.6)
    ax.set_xlabel("Tiempo (s)"); ax.set_ylabel(f"sin deriva ({UNIDAD})")
    ax.legend(fontsize=8, loc="upper right"); ax.grid(alpha=0.3)
    ax.set_title("Eventos detectados")
    plt.tight_layout(); plt.show()


## 10. Elección del umbral y verificación: ¿son eventos reales o ruido?

**Entra:** la señal sin deriva. **Sale:** el `k` elegido, la curva de estabilidad y el control de
falsos positivos.

Se barre un rango de umbrales `k` (en múltiplos del ruido) y se cuenta cuántos eventos sobreviven
a cada uno. Como control, se corre el **mismo detector sobre la señal invertida**: una contracción
sólo puede ir en un sentido, así que todo lo que aparece del lado invertido es ruido.

- si el conteo se mantiene en una **meseta** a lo largo de varios `k` y los falsos son **0**, los
  eventos son reales y el resultado no depende del umbral elegido;
- si **decae monótonamente** y los falsos acompañan al conteo real, lo que se está contando es
  ruido, y **el conteo no se reporta**.

### La elección de `k` ya es automática (v4)

`cr.elegir_k_meseta()` aplica esa regla sola. Dos decisiones que no son obvias y que costó
encontrar:

**Cuál meseta, cuando hay varias: la de `k` más bajo.** Al subir el umbral se van perdiendo
eventos reales, así que la primera meseta con 0 falsos es la que ya eliminó el ruido y todavía no
empezó a comerse señal. La regla "la meseta más larga" da la respuesta **equivocada**: en
Video_063 hay una meseta de 6 eventos en k=6..8 (2 puntos) y otra de 5 en k=10..20 (4 puntos), y
la validada es la de 6.

**Filtrar los falsos ANTES de buscar el tramo constante, no después.** Agrupar por conteo y
después exigir 0 falsos en todo el grupo es demasiado estricto: un tramo de conteo constante suele
empezar unos `k` antes de que los falsos lleguen a cero. En Video_466 el tramo de 5 eventos empieza
en k=6, donde todavía hay 1 falso, y eso invalidaba la meseta real (k=8..15) entera.

Hay un test de regresión con los diez escaneos reales y su respuesta validada:
`python tests/test_seleccion_k.py`.

### Por qué importa

El fallo del `k` por defecto **no era ruidoso**. En Video_583, con `k=8` salían 9 eventos (1
falso) y `rhythm_split` **no encontraba el tren estimulado**, porque los eventos espurios
ensuciaban el ajuste de la grilla. Con `k=12`, que es donde está la meseta, el tren aparece limpio:
5/5 ranuras, CV 0.4 %. No avisaba nada: simplemente hacía desaparecer el tren.

> **Chequeo de aceptación 3.** Si no hay meseta, el reporte marca el conteo `[NO REPORTABLE]` y
> graba `conteo_reportable = False`. No se baja `k` para "encontrar" eventos.

### Qué más cambia el veredicto

**La ventana del detrend.** Desde la Fase 2.2 el conteo se repite con 0.75×, 1× y 1.5× la ventana
automática, y solo es reportable si da lo mismo en las tres (`conteo_por_ventana` en el reporte). Antes,
con la ventana fija de 2 s, Video_491 no tenía meseta porque la mediana se comía sus eventos de ~1 s.

**Más de una meseta.** Se usa la de `k` más bajo (cuenta todos los eventos; las de `k` más alto cuentan
solo los grandes), pero el reporte las lista todas y la figura 05 las marca en naranja. Video_063 tiene
dos: 6 eventos (k 5.3–8.6) y 5 (k 9.4–24); la diferencia es el evento débil de t = 0.31 s (H11). El `k`
usado es el del centro de la meseta, no el borde donde empiezan los falsos.

Este
cuaderno **ya no** incluye la "segunda opinión" del motor original: daba otros conteos (en Video_466
marcaba "dudoso" mientras la meseta tenía 0 falsos) y no había criterio para decidir a cuál creer.

**Ejes de `05_estabilidad_umbral.png`:** `x` = k, `y` = número de eventos; en azul los eventos de la señal y en rojo los falsos de
control (mismo detector sobre la señal invertida); la banda verde es la meseta elegida. Lo que se busca
es una zona horizontal del azul con el rojo en 0.


In [ ]:
# Escaneo de estabilidad del reporte (el mismo que eligió el k de la sección 9).
estab = a_rep["estabilidad"]
display(estab.round(4))

sel = cr.elegir_k_meseta(estab)
print(sel["motivo"])
if sel["hay_meseta"]:
    print(f"CHEQUEO 3 - meseta con 0 falsos: OK -> {sel['n_eventos']} eventos reportables "
          f"(k dentro de {sel['k_rango'][0]:g}..{sel['k_rango'][1]:g}; se usó k={K_USADO:g})")
else:
    print("CHEQUEO 3 - SIN MESETA: el conteo NO se reporta.")
    print("  Los eventos detectados sirven para auditar el gráfico, no para tabular.")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(estab.k, estab.eventos, "o-", color="#1f77b4", label="eventos")
ax.plot(estab.k, estab.falsos_control, "s--", color="crimson", label="falsos de control (señal invertida)")
if sel["hay_meseta"]:
    ax.axvspan(sel["k_rango"][0], sel["k_rango"][1], color="green", alpha=0.12, label="meseta elegida")
ax.axvline(K_USADO, color="black", lw=0.8, ls=":", label=f"k usado = {K_USADO:g}")
ax.set_xlabel("k (umbral = k × ruido)"); ax.set_ylabel("número de eventos")
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_title("Estabilidad del conteo respecto del umbral")
plt.tight_layout()
fig.savefig(OUT_DIR / "05_estabilidad_umbral.png", dpi=140)
plt.show()


## 11. Adelgazamiento y poblaciones de eventos

**Entra:** los eventos y el canal de grosor. **Sale:** cuánto adelgaza realmente el gel, y si hay
más de un tipo de contracción en el vídeo.

### 11a. Promedio de eventos alineados

El grosor sigue siendo la variable biomecánicamente interesante: es la que se relaciona con la
deformación del material. Lo que no sirve es usarla para *detectar*. La forma de medirla cuando
está por debajo del ruido por fotograma es **alinear todos los eventos en su pico y promediarlos**:
el ruido baja como √N mientras la señal se mantiene. Con 5 eventos, el adelgazamiento de Video_063
pasó de ~1 σ por fotograma a 11 σ.

Se reportan **dos medidas**, y hay que usar la robusta:

- **`mínimo`** — el mínimo del promedio alineado. Puede estar contaminado.
- **`robusto`** — promedio de los 3 fotogramas **posteriores** al pico.

El motivo es un artefacto real: en el fotograma de máxima velocidad, el grosor medido da un salto
**positivo** (+0.39 px en Video_063). Eso no es engrosamiento, es **motion blur** — el borde se
emborrona por el movimiento y los dos bordes se "abren". El cuaderno lo detecta y avisa.

**Ejes:** `x` = tiempo respecto del pico del evento (s, 0 = pico); eje `y` izquierdo (azul) =
traslación promedio (px); eje `y` derecho (rojo) = cambio de grosor promedio (px). Los dos ejes
tienen escalas distintas a propósito: el punto es comparar la **forma**, no la magnitud.

### 11b. Poblaciones

Un mismo vídeo puede tener contracciones **espontáneas** (chicas y rápidas) y **estimuladas**
(grandes y lentas). Promediarlas juntas da una amplitud y una frecuencia que no describen a
ninguna de las dos. Si las amplitudes se separan en dos grupos (razón de medianas ≥ 2), se
reportan por separado.

In [ ]:
if a_rep["n_eventos"] == 0 or a_rep.get("_lag") is None or not a_rep["_n_prom"]:
    print(">>> No se detectaron eventos: nada que promediar. Revisar la meseta de la sección 10.")
else:
    lag, prom_g, prom_c = a_rep["_lag"], a_rep["_prom_g"], a_rep["_prom_c"]
    n_prom = a_rep["_n_prom"]
    mg = a_rep["ruido_grosor_px"]
    ruido_prom = mg / np.sqrt(max(n_prom, 1))
    pico_c = float(np.max(prom_c))

    print(f"Eventos promediados: {n_prom}")
    print(f"Ruido del grosor por fotograma : {mg:.4f} px")
    print(f"Ruido del promedio (MAD/sqrt N): {ruido_prom:.4f} px")
    print(f"Traslación en el pico          : {pico_c:.3f} px")
    print(f"Adelgazamiento  mínimo         : {a_rep['adelgazamiento_px']:.4f} px "
          f"({a_rep['adelgazamiento_sigma']:.1f} sigma, a {a_rep['retardo_adelgazamiento_s']:+.2f} s del pico)"
          f"  =  {a_rep['cociente_adelg_trasl_pct']:.1f} % de la traslación")
    if "adelgazamiento_robusto_px" in a_rep:
        print(f"Adelgazamiento  ROBUSTO        : {a_rep['adelgazamiento_robusto_px']:.4f} px "
              f"({a_rep['adelgazamiento_robusto_sigma']:.1f} sigma)"
              f"  =  {a_rep['cociente_robusto_pct']:.1f} % de la traslación   <-- usar éste")
    if "blur_sigma" in a_rep:
        print(f"\nAVISO: el grosor da un salto POSITIVO de {a_rep['blur_px']:.3f} px "
              f"({a_rep['blur_sigma']:.1f} sigma) en el fotograma más rápido. No es "
              f"engrosamiento: es motion blur. Usar la medida robusta.")

    fig, (axA, axB) = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1, 2]})
    axA.plot(lag, prom_c, color="steelblue", lw=1.6, label="traslación")
    axAb = axA.twinx()
    axAb.plot(lag, prom_g, color="crimson", lw=1.6, label="grosor")
    axAb.axhline(0, color="crimson", lw=0.6, ls=":")
    axA.axvline(0, color="gray", lw=0.8)
    axA.set_xlabel("t respecto del pico (s)")
    axA.set_ylabel(f"traslación ({UNIDAD})", color="steelblue")
    axAb.set_ylabel(f"grosor ({UNIDAD})", color="crimson")
    axA.set_title(f"Promedio de {n_prom} eventos alineados", fontsize=10)

    axB.scatter(eventos.tiempo_s, eventos.amplitud_px, s=30, color="crimson", zorder=5)
    axB.vlines(eventos.tiempo_s, 0, eventos.amplitud_px, color="steelblue", lw=1)
    axB.set_xlabel("Tiempo (s)"); axB.set_ylabel(f"Amplitud ({UNIDAD})")
    axB.set_title("Amplitud por evento", fontsize=10); axB.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    if a_rep.get("poblaciones"):
        print("\nDOS POBLACIONES de eventos, separadas por amplitud:")
        display(pd.DataFrame(a_rep["poblaciones"]).round(3))
    else:
        print("\nUna sola población de eventos (las amplitudes no se separan en dos grupos).")


## 11c. Cinética: TTP, RT50 y amplitud relativa

**Entra:** los eventos de la sección 9. **Sale:** por evento, el inicio, el tiempo al pico (TTP), el tiempo
de relajación al 50 % (RT50) y la amplitud relativa; y el resumen por vídeo.

**Qué eventos usa.** Los de la sección 9 (`a_rep`): son exactamente los de `contracciones.xlsx`.
La cinética se calcula dentro de `cr.analizar` y acá sólo se muestra.

**Definiciones** (sobre el canal de detección sin deriva, con el reposo en 0; `A` = amplitud):

- **inicio**: cruce del 10 % de `A` antes del pico. **TTP**: del inicio al pico.
- **RT50**: del pico al cruce del 50 % de `A` en la bajada. *No* es la mitad de la duración de la
  relajación, que es lo que calculaba el script de MATLAB.
- **amplitud relativa**: `A` / grosor en reposo, en %. No depende del aumento, así que se puede
  comparar entre vídeos sin calibrar.

**Por qué a veces sale una cota y no un valor.** La cámara toma una imagen cada 33 ms. Un cruce
entre dos imágenes pudo ocurrir en cualquier punto entre ellas, y el pico verdadero puede estar
hasta una imagen antes o después de la más alta. Si la subida ocupa 2 imágenes, el TTP verdadero
está entre 0 y 3 intervalos: lo único que se puede afirmar es **TTP < 100 ms**. Por eso cada
evento trae su intervalo `[min, max]`, y el valor se reporta solo si la subida (o la bajada al
50 %) ocupa al menos **5 fotogramas** y el conteo de eventos es reportable. En las muestras lentas
el pico además es una **meseta** de varios fotogramas (`meseta_pico_frames`), y eso ensancha el
intervalo.

**Ejes de `11_cinetica_*.png`:** izquierda — `x` = **fotogramas** respecto del pico (no segundos,
para que se vea cuántas muestras tiene la subida), `y` = señal / amplitud; gris cada evento, azul
la mediana, punteadas en el 10 % y el 50 %. Derecha — `x` = evento, `y` = ms: TTP (azul) y RT50
(rojo) de cada evento con su intervalo; la franja gris es la zona de menos de 5 fotogramas.

**Ojo con el resumen por vídeo.** La mediana que resume el vídeo mezcla las dos poblaciones (estimuladas
y espontáneas). En Video_prueba, por ejemplo, la amplitud relativa del resumen (0.71 %) es la de las
espontáneas (21 de 28 eventos); la de las estimuladas es 2.29 %. La tabla de abajo la separa por grupo.


In [ ]:
# La cinética ya está calculada en a_rep (cr.analizar): los números son los de contracciones.xlsx.
print(f"Eventos: {a_rep['n_eventos']}")
cr.imprimir_cinetica(a_rep)

tabla_cin = a_rep["_cinetica"]
if len(tabla_cin):
    display(tabla_cin[["evento"] + (["grupo"] if "grupo" in tabla_cin else []) +
                      ["tiempo_s", "amplitud_px", "amplitud_relativa_pct", "meseta_pico_frames",
                       "ttp_frames", "ttp_s", "ttp_min_s", "ttp_max_s", "ttp_medible",
                       "rt50_frames", "rt50_s", "rt50_min_s", "rt50_max_s", "rt50_medible"]].round(3))

    if "grupo" in tabla_cin:
        por_grupo = (tabla_cin.groupby("grupo")
                     .agg(n=("evento", "size"),
                          amplitud_mediana_px=("amplitud_px", "median"),
                          amplitud_relativa_mediana_pct=("amplitud_relativa_pct", "median"))
                     .round(3))
        print("\nPor grupo (el resumen de arriba mezcla los grupos):")
        display(por_grupo)

    png_cin = cr.graficar_cinetica([(VIDEO.stem, a_rep)], OUT_DIR / f"11_cinetica_{VIDEO.stem}.png")
    display(Image(str(png_cin)))


## 12. Ritmo: enganche de fase, estimuladas vs espontáneas

**Entra:** los eventos de la sección 9 (`a_rep["ritmo"]`). **Sale:** qué contracciones son del estimulador y cuáles
espontáneas, el período del tren, y el contraste contra la frecuencia configurada.

### El criterio: enganche de fase, no amplitud

`src/rhythm_split.py` **no usa la amplitud** (con estimulación débil los grupos se solapan) ni una
ventana temporal (las espontáneas siguen apareciendo durante el tren). Usa la única propiedad que
define a una contracción estimulada: **está enganchada en fase a un reloj periódico**. El
estimulador dispara en `t = fase + n·T`; se busca la grilla `(T, fase)` que mejor explica un
subconjunto de los eventos, y lo que queda afuera es espontáneo.

Como la amplitud **no interviene** en la clasificación, que los grupos resulten tener amplitudes
distintas es **evidencia independiente** de que la separación es real.

El p-valor sale de un Monte Carlo que corre el procedimiento completo sobre datos con los
intervalos permutados: conserva la distribución de intervalos y destruye el enganche de fase, así
que ya incluye el haber probado muchos períodos.

### Tres problemas que hubo que resolver

- **Armónicos.** Si el período real es T, la grilla de T/2 también contiene todos los eventos
  verdaderos, pero con la mitad de las ranuras vacías; como tiene más ranuras puede acumular un
  z mayor. Se exige `min_captura` (75 %) y, entre candidatos que puntúan casi igual, se elige el
  período **más largo**.
- **Intrusos con brazo de palanca.** Un espontáneo tomado como primera ranura movía el período y
  repartía su error entre todos los residuos. Se ajusta primero con **Theil-Sen** y recién al
  final por mínimos cuadrados sobre el conjunto ya limpio.
- **Error estándar cero.** El tren cae siempre en el mismo número de fotogramas, así que los
  residuos daban exactamente 0. Se pone como piso el error de cuantización, `resolución/√12`.

### El resultado sobre los cinco vídeos estimulados

Con la base de tiempo por PTS, los cinco dan una frecuencia **indistinguible de los 0.1 Hz
configurados** — antes, con el fps declarado, Video_prueba daba −0.9 % y Video_063 −0.43 %:

| vídeo | frecuencia medida | diferencia | jitter | captura |
|---|---|---|---|---|
| Video_prueba | 0.09993 ± 0.000041 Hz | −0.07 % | 20.9 ms | 7/7 |
| Video_063 | 0.10001 ± 0.000030 Hz | +0.01 % | 9.6 ms | 5/5 |
| Video_268 | 0.10000 ± 0.000023 Hz | 0.00 % | 9.6 ms | 6/6 |
| Video_466 | 0.10006 ± 0.000120 Hz | +0.06 % | 35.1 ms | 5/5 |
| Video_583 | 0.09999 ± 0.000067 Hz | −0.01 % | 25.4 ms | 6/6 |

### Latidos dudosos

Un latido del tren cuyo instante detectado se corrió más que la tolerancia **no se descarta ni se
mezcla con las espontáneas**: se reporta aparte como `estimulados_dudosos` y no entra en el ajuste
del período. Para decidir qué es, se compara su amplitud con la de los estimulados — y como la
amplitud no se usó para clasificar, esa comparación es independiente.

### Límites

- Necesita al menos **4 latidos estimulados**.
- Una serie espontánea **muy** regular es indistinguible de una estimulada por los tiempos solos.
- Si la estimulación **cambia de frecuencia a mitad** del vídeo, esta versión encuentra un solo
  tren. Habría que correrla por tramos.
- Con **4 o 5 eventos no se corre el Monte Carlo** (hacen falta al menos 6): el p-valor queda `NaN` y el tren
  se acepta por la captura (≥ 75 %) y el puntaje z. Es el caso de Video_466 (5 eventos).
- Con 200 simulaciones, el p mínimo posible es 1/201 ≈ 0.005: cuatro de los cinco vídeos dan exactamente ese
  valor, que sólo dice "ninguna simulación llegó al z observado", no cuán fuerte es la evidencia.
- La significancia no está calibrada para actividad espontánea agrupada o muy regular.


In [ ]:
rit = a_rep.get("ritmo")
if a_rep["n_eventos"] < 4 or rit is None:
    print(">>> Hacen falta al menos 4 eventos para buscar un tren estimulado "
          f"(hay {a_rep['n_eventos']}). Revisar la meseta de la sección 10.")
else:
    if not rit["hay_estimulacion"]:
        print(f"No se detectó un tren periódico: todos los eventos se toman como espontáneos. "
              f"({rit.get('motivo', '')})")
    else:
        p = rit["p_valor"]
        p_txt = f"{p:.4f}" if np.isfinite(p) else "no se calculó (menos de 6 eventos)"
        print(f"TREN ESTIMULADO (enganche de fase): z = {rit['z']:.1f}, p = {p_txt}")
        print(f"   período    : {rit['periodo_s']:.5f} ± {rit['periodo_err_s']:.5f} s")
        print(f"   frecuencia : {rit['frecuencia_Hz']:.5f} ± {rit['frecuencia_err_Hz']:.6f} Hz")
        print(f"   jitter     : {rit['jitter_s']*1000:.1f} ms"
              + ("  (por debajo de un fotograma: es el piso de resolución)"
                 if rit.get("jitter_limitado_por_resolucion") else ""))
        print(f"   captura    : {rit['tasa_captura_pct']:.0f} %  ({rit['n_estimulados']}/{rit['n_ranuras']} "
              f"ranuras), tren de {rit['tren_inicio_s']:.1f} a {rit['tren_fin_s']:.1f} s")
        if rit.get("n_estimulados_dudosos"):
            print(f"   DUDOSOS    : {rit['n_estimulados_dudosos']} evento(s) cerca de una ranura pero fuera de "
                  f"tolerancia. Comparar su amplitud con la de los estimulados.")

    print("\nGrupos (la amplitud NO se usó para clasificar; que difiera es evidencia aparte):")
    if rit.get("resumen_grupos") is not None:
        display(rit["resumen_grupos"])
    if rit["hay_estimulacion"] and len(rit.get("grilla", [])):
        print("\nGrilla del tren (ranura esperada contra latido medido):")
        display(rit["grilla"].round(4))
        if FREQ_ESTIMULO:
            comp = cr.rs.comparar_con_equipo(rit, FREQ_ESTIMULO, fps_nominal=a_rep.get("fps_medido"))
            print(f"\nContraste contra la frecuencia configurada ({FREQ_ESTIMULO:g} Hz):")
            display(pd.DataFrame({"valor": comp}))

    png_ritmo = cr.graficar_ritmo([(VIDEO.stem, a_rep)], OUT_DIR / f"10_ritmo_{VIDEO.stem}.png")
    if png_ritmo:
        display(Image(str(png_ritmo)))


## 13. Exportación de resultados

**Entra:** todo lo anterior. **Sale:** los archivos que quedan en
`data/processed_data/<nombre_del_vídeo>/`.

| archivo | contenido |
|---|---|
| `serie_temporal.xlsx` | hoja `diagnostics`: una fila por fotograma con los 4 canales, residuos, `outlier_frac` y etiqueta de calidad. Hoja `resumen`: parámetros usados y estadísticos globales. **Para una corrida que se vaya a archivar, usar `main.py`: su hoja `resumen` registra más parámetros** |
| `contracciones.xlsx` | el reporte de `contraction_report.py`: `estab_*`, `resumen_*`, `eventos_*`, `ritmo_*`, `grilla_*`, `cinetica_*` y `poblac_*`. **Lo genera el propio script** (esta celda lo llama), así que no hay un formato paralelo |
| `00_max_projection.png`, `00_roi_profile.png` | el mapa de máxima intensidad y el perfil de la ROI |
| `01_serie_temporal.png` | las series de la sección 6 |
| `05_estabilidad_umbral.png` | la curva de estabilidad de la sección 10 |
| `09_contracciones_<vídeo>.png`, `10_ritmo_<vídeo>.png`, `11_cinetica_<vídeo>.png` | los eventos, el ritmo y la cinética (los genera el script) |

El cuaderno ya no produce `eventos.xlsx` ni las figuras `02`, `03`, `04` y `06`: eran del motor
original, que se retiró de este flujo.


In [ ]:
import subprocess
from src.qc_visualization import _write_xlsx

_write_xlsx(df, OUT_DIR / "serie_temporal.xlsx", summary={
    "video"                 : VIDEO.name,
    "fotogramas"            : len(df),
    "px_to_mm"              : PX_TO_MM,
    "canal_de_deteccion"    : CANAL_USADO,
    "ROI x_start"           : roi["x_start"],
    "ROI x_end"             : roi["x_end"],
    "ROI metodo"            : q.get("method"),
    "ransac_degree"         : CFG.ransac_degree,
    "ransac_umbral"         : CFG.ransac_residual_threshold or "adaptativo",
    "use_clahe"             : CFG.use_clahe,
    "base_tiempo"           : BASE_TIEMPO,
    "fps_serie"             : round(fps, 4),
    "grosor medio (px)"     : round(float(df.thickness_px.mean()), 3),
    "recorrido centro (px)" : round(float(df.center_px.max() - df.center_px.min()), 3),
})

# El reporte se genera con el propio script, con los mismos parámetros que usó este cuaderno.
cmd = [sys.executable, str(PROJECT_ROOT / "scripts" / "contraction_report.py"),
       "--input", str(OUT_DIR / "serie_temporal.xlsx"), "--canal", CANAL_USADO,
       "--k", "auto" if AMP_K is None else str(AMP_K),
       "--win-s", "auto" if WIN_DERIVA_S is None else str(WIN_DERIVA_S), "--half-s", str(HALF_EVENTO_S),
       "--output-dir", str(OUT_DIR)]
if SEP_S:
    cmd += ["--sep-s", str(SEP_S)]
if FREQ_ESTIMULO:
    cmd += ["--frecuencia-estimulo", str(FREQ_ESTIMULO)]
salida = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))
if salida.returncode != 0:
    print("FALLÓ contraction_report.py:\n", salida.stderr[-2000:])
else:
    # Comprobación cruzada: el reporte del archivo tiene que coincidir con lo que mostró el cuaderno.
    hojas = pd.read_excel(OUT_DIR / "contracciones.xlsx", sheet_name=None)
    resumen_hoja = next(v for k_, v in hojas.items() if k_.startswith("resumen_"))
    n_archivo = int(resumen_hoja["n_eventos"].iloc[0])
    estado = "COINCIDE" if n_archivo == a_rep["n_eventos"] else "DIFIERE"
    print(f"Eventos en contracciones.xlsx: {n_archivo}  |  en el cuaderno: {a_rep['n_eventos']}  ->  {estado}")

print("\nArchivos generados en", OUT_DIR.relative_to(PROJECT_ROOT))
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name:<34} {f.stat().st_size/1024:8.1f} kB")


---

## Correspondencia con MATLAB (`Contraction_Analysis.mlx`) y con MuscleMotion

El live script de Tecnun parte de un `contraction.txt` de dos columnas (tiempo, amplitud) que ya
existe — y ese archivo es **la salida de MuscleMotion**. Este cuaderno cubre ese mismo tramo **y
el que lo precede**: la construcción de la señal a partir del vídeo.

| `Contraction_Analysis.mlx` | Este cuaderno |
|---|---|
| 1. Importación de `contraction.txt` y recorte de artefactos | 2–6. El vídeo se convierte en la señal, midiendo los bordes fotograma a fotograma |
| 1. `detrend` lineal para centrar la señal | 8. Percentil móvil al 90 %: sigue la deriva lenta en lugar de suponerla lineal |
| 2. `findpeaks` con `MinPeakDistance` fijo | 9. `find_peaks` con distancia derivada del ancho de evento medido en los datos |
| 2. Basal por mediana móvil (`Window_Size` fijo) | 8. Basal por percentil móvil, con ventana derivada del ancho de evento |
| 3. Inicio y fin de cada pico por cruce con la basal | 9. Duración a media altura sobre la señal de profundidad |
| 4. Métricas por pico (TTP, RT50, amplitud, P2P) | **11c.** TTP, RT50 y amplitud relativa, con cota cuando no son medibles; P2P = `intervalo_s` |
| — | 7. Elección del observable con datos |
| — | **10. Elección automática de `k` y verificación estadística del umbral** |
| — | 11. Adelgazamiento por promedio de eventos alineados |
| — | **12. Separación estimuladas/espontáneas por enganche de fase** |
| — | 3–5. Control de calidad visual del borde |

### Seis cosas del script de MATLAB a revisar antes de portarlo

Detalle completo, con el arreglo de cada una, en `claude/revision-script-matlab.md`.

1. **No hay umbral de amplitud.** `findpeaks` se llama sólo con `MinPeakDistance`, sin
   `MinPeakProminence`, aunque la descripción dice que sí. Devuelve el máximo local de cada
   ventana **siempre**, haya contracción o no.
2. **`Half_Relax_Time` no calcula RT50.** Hace `(Time(Off)−Time(P))*0.5`, que es la mitad de la
   duración total de la relajación, no el tiempo hasta caer al 50 % de la amplitud.
3. **El cambio relativo divide por la señal centrada en cero** (`Amplitude_Detrend(On)`), o sea
   por ruido. El porcentaje sale inestable y puede cambiar de signo sin que pase nada físico.
4. **`Idx_Before = Idx_Before − 5`**, sin justificación: 167 ms sumados a cada TTP.
5. **`Min_Peak_Distance = 520` hardcodeado** (la versión paramétrica está comentada justo arriba):
   a 30 fps son 17.3 s, fusiona latidos con estimulación a 0.1 Hz.
6. **`Data(1:2000,:)` descarta el final de los archivos largos.** Puede ser intencional si se sabe
   cuántos segundos se necesitan, pero conviene recortar por tiempo y no por número de muestra,
   para que no dependa del vídeo.

### El límite de fondo: el muestreo es más lento que el evento

Esto no es del script: afecta a cualquier implementación. Medido sobre el promedio de eventos
alineados de cada vídeo (ruido del promedio 0.03 px):

| vídeo | TTP | subida 10–90 % |
|---|---|---|
| Video_prueba | 67 ms (**2 fotogramas**) | 33 ms (**1 fotograma**) |
| Video_063 | 67 ms (**2 fotogramas**) | 33 ms (**1 fotograma**) |
| Video_268 | 67 ms (**2 fotogramas**) | 33 ms (**1 fotograma**) |
| Video_466 | 300 ms (9 fr) | 100 ms (3 fr) |
| Video_583 | 300 ms (9 fr) | 133 ms (4 fr) |

El evento promedio de Video_prueba, en múltiplos del ruido del promedio:

    fotograma:  -3   -2   -1    0   +1   +2   +3   +4
    señal:      -0    1  147  190  146   77   31   10

Pasa de ruido a 147 σ **en un fotograma**. Ahí un TTP de "67 ms" no describe la muestra: describe
la cámara. Y ni siquiera 67 ms es una cota segura: el pico verdadero puede estar hasta un fotograma
después del muestreado, así que lo que se puede afirmar es **TTP < 100 ms**.

Es coherente con la fisiología: el twitch de músculo esquelético tiene un TTP típico de 30–80 ms y
la cámara muestrea cada 33 ms. **Para medir TTP y RT50 de verdad hacen falta 200–300 fps.**

**Implementado** en `src/cinetica.py` (sección 11c): onset/pico/offset sobre la detección ya
validada, RT50 como la caída al 50 % de la amplitud, y la métrica marcada como no medible, con su
cota, cuando la subida dura menos de 5 fotogramas.

### Frente a MuscleMotion

La ventaja de medir **geometría de borde** en vez de **intensidad** ya no es sólo un argumento.
Detalle en `claude/comparacion-musclemotion.md`; en resumen:

**Su escala temporal está comprimida.** ImageJ importa menos fotogramas que OpenCV en los cinco
vídeos (déficit de 0.65 % a 4.05 % **según el archivo**, y OpenCV coincide exacto con el conteo de
paquetes de `ffprobe`). Como MuscleMotion arma su eje como `fotograma / 30`, ese déficit comprime
su tiempo por un factor que cambia de vídeo en vídeo, así que **no se cancela al comparar
muestras**.

**Sobre-detecta y no tiene con qué saberlo.** En Video_063 reporta 35 picos donde nosotros
certificamos 6, y 19 de esos 35 llevan un aviso de sus propios chequeos internos. Dos tests:

- En **nuestra** señal (SNR 44 por fotograma), la amplitud en los instantes de sus picos extra
  **no está elevada** respecto de instantes al azar (cociente 0.62–0.86, p = 0.79–0.95). Ahí no se
  movió nada medible.
- **Su propia señal**, sometida a nuestro escaneo de estabilidad, converge a 6 eventos en
  Video_063 y 6–8 en Video_268 — el mismo número que nosotros, midiendo una cosa distinta.

Lo decisivo no es que se equivoque más, sino que su salida **no trae un control de falsos
positivos**: sus 35 picos y sus 6 picos se leen igual.

### Lo que todavía falta medir

Un **vídeo de control donde sólo cambie la luz y el gel esté quieto**: mismo campo, mismo gel,
inmóvil, con un cambio de iluminación gradual o un parpadeo. Sobre ese vídeo este flujo debería
dar una serie plana. Es lo único barato que falta para cerrar el argumento de que medir geometría
es inmune a la iluminación.
